# EMRI waveform comparison — JAX vs FEW

Three pipelines on the same physical parameter set:

1. **JAX cubic** — fully JAX: fewtrax trajectory + JAX amplitudes + JAX mode selection + waveform built with cubic-spline interpolation of phases (no FEW state needed; differentiable end-to-end).
2. **JAX DOPR** — FEW trajectory (for its DOPR853 dense-output Bezier phase spline) + JAX amplitudes + JAX mode selection + waveform built with the DOPR853 phase spline.
3. **FEW** — `GenerateEMRIWaveform` reference.

For each of `N = 500` parameter sets drawn from physical priors, we measure wall time of each pipeline and the mismatch of the JAX pipelines against the FEW reference, for both polarisations.  Waveforms are **2 years** long at `dt = 10 s`.


In [1]:
import os
idx = 1
os.environ["CUDA_VISIBLE_DEVICES"] = str(idx)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["FEW_DATA_DIR"] = "/home/fedefant/.local/share/few/v2.0.0/download"


In [2]:
import time
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

import few
from few.waveform import GenerateEMRIWaveform
from few.waveform.waveform import FastKerrEccentricEquatorialFlux
from few.utils.constants import MTSUN_SI, YRSID_SI
from few.utils.utility import get_mismatch, get_p_at_t

Gpc      = few.utils.constants.Gpc
MRSUN_SI = few.utils.constants.MRSUN_SI

print("JAX backend:", jax.default_backend(), "  devices:", jax.devices())


JAX backend: gpu   devices: [CudaDevice(id=0)]


In [3]:
# FEW model: provides amplitude_generator, ylm_gen, mode_selector and few_traj.
cfg = few.get_config_setter(reset=True)
cfg.set_log_level("warning")

try:
    model = FastKerrEccentricEquatorialFlux(force_backend='gpu')
    print("FEW model loaded on GPU.")
except Exception as exc:
    print(f"GPU init failed ({exc}); falling back to CPU.")
    model = FastKerrEccentricEquatorialFlux(force_backend='cpu')
few_traj = model.inspiral_generator

mode_selection_threshold = 1e-5
mode_selector_kwargs = dict(
    mode_selection_threshold=mode_selection_threshold,
    include_minus_mkn=True,
)

# Reference waveform generator (frame='source' so we get raw h+, hx without
# LISA response — same conventions as cell ff7b89cf of EMRI_wf_JAX_updated).
waveform_gen = GenerateEMRIWaveform(
    "FastKerrEccentricEquatorialFlux",
    sum_kwargs=dict(pad_output=True),
    force_backend="gpu",
    frame="source",
    return_list=True,
    mode_selection_threshold=mode_selection_threshold,
    include_minus_mkn=True,
)


FEW model loaded on GPU.


In [4]:
# fewtrax: JAX trajectory + JAX amplitudes + FEW-compatible mode selector.
from fewtrax.data import load_flux_data
from fewtrax.trajectory import EMRIInspiral
from fewtrax.amplitude import JAXAmplitudeInterpolator
from fewtrax.data.loader import load_amplitude_data_jax

FEW_DATA_DIR = os.environ["FEW_DATA_DIR"]

flux_data = load_flux_data(FEW_DATA_DIR)
traj_jax  = EMRIInspiral(flux_data)

# Full (all 6993 modes) JAX amplitude module — ~5 GB on GPU.  Loaded once;
# select_modes_few then folds into a single jitted GPU kernel.
amp_data_full = load_amplitude_data_jax(FEW_DATA_DIR, mode_indices=None)
amp_jax_full  = JAXAmplitudeInterpolator(amp_data_full)
print(f"fewtrax loaded: {amp_data_full.n_modes} modes on device.")

l_full = np.asarray(amp_data_full.l_arr)
m_full = np.asarray(amp_data_full.m_arr)
n_full = np.asarray(amp_data_full.n_arr)


fewtrax loaded: 6993 modes on device.


In [5]:
# Stealed from FEW
def _get_viewing_angles(qS, phiS, qK, phiK):
        """Transform from the detector frame to the source frame"""

        cqS = np.cos(qS)
        sqS = np.sin(qS)

        cphiS = np.cos(phiS)
        sphiS = np.sin(phiS)

        cqK = np.cos(qK)
        sqK = np.sin(qK)

        cphiK = np.cos(phiK)
        sphiK = np.sin(phiK)

        # sky location vector
        R = np.array([sqS * cphiS, sqS * sphiS, cqS])

        # spin vector
        S = np.array([sqK * cphiK, sqK * sphiK, cqK])

        # get viewing angles
        phi = -np.pi / 2.0  # by definition of source frame

        theta = np.arccos(-np.dot(R, S))  # normalized vector

        return (theta, phi)


---
## Waveform builder

Identical to `Interp_and_modesum_WFbuilder` from [EMRI_wf_JAX_updated.ipynb] cell `67e78dcd`.  
**Cubic phase path**: cubic spline of `Phi_phi, Phi_theta, Phi_r` on the sparse trajectory — does *not* need any FEW state, so the whole pipeline is JAX-traceable.  
**DOPR phase path**: pass `few_traj.integrator_spline_t` and `few_traj.integrator_spline_phase_coeff` and the builder evaluates FEW's DOPR853 dense-output degree-7 Bezier polynomial.


In [6]:
# Mode-batched interp + direct mode sum.
# v3: cubic spline interp (amplitudes AND phases) + factorised exp [ACTIVE]
#
# v3 matches FEW's InterpolatedModeSum accuracy (not-a-knot cubic splines)
# while keeping the exp-factorisation speedup from v2.

from scipy.interpolate import CubicSpline as _CubicSpline

def _fit_spline(t_sparse_np, y_sparse_np):
    """Fit a not-a-knot cubic spline and return scipy's (4, n-1, ...) coefficient array."""
    cs = _CubicSpline(t_sparse_np, y_sparse_np)
    return jnp.asarray(cs.c)   # (4, n_sparse-1, *y_shape[1:])


@jax.jit
def _bracket(t_traj, t_dense):
    """Bracket indices and normalised weights: inds, w = (t-t_i)/(t_{i+1}-t_i)."""
    inds = jnp.clip(
        jnp.searchsorted(t_traj, t_dense, side="right") - 1,
        0, t_traj.size - 2,
    )
    w = (t_dense - t_traj[inds]) / (t_traj[inds + 1] - t_traj[inds])
    return inds, w


@jax.jit
def _eval_cubic(spline_c, inds, dx):
    """Evaluate cubic splines at dense points via Horner's method.

    spline_c : (4, n_sparse-1, n_splines)  scipy cs.c layout:
                 c[0]=cubic, c[1]=quad, c[2]=linear, c[3]=constant
    inds     : (n_dense,) int  -- bracket indices
    dx       : (n_dense,) float -- time offset within bracket (t - t_i)
    returns  : (n_dense, n_splines)
    """
    result = spline_c[0, inds]                              # d (cubic coeff)
    result = spline_c[1, inds] + dx[:, None] * result      # c + dx*d
    result = spline_c[2, inds] + dx[:, None] * result      # b + dx*(c+dx*d)
    result = spline_c[3, inds] + dx[:, None] * result      # a + dx*(...)
    return result                                            # (n_dense, n_splines)

# =============================================================================
# v3 -- cubic spline interp (amplitudes + phases) + factorised exp    [ACTIVE]
# =============================================================================
# Cubic splines (not-a-knot, same as FEW's InterpolatedModeSum) are fitted
# once to the 67 sparse trajectory points -- negligible CPU time.
# Evaluation uses Horner's method: each step gathers from a tiny
# (66, batch) coefficient array before writing the (n_t_dense, batch) result.

@jax.jit
def _interp_modesum_batch_cubic(
    inds, dx,            # (n_t_dense,) -- bracket index and time offset
    spline_coeffs_b,     # (4, n_sparse-1, batch) complex128 -- tiny coefficient batch
    ylm_pos_b, coeff_neg_b,
    E_phi,               # (n_uniq_m, n_t_dense) -- pre-factored m-phase exp table
    E_theta,             # (n_uniq_k, n_t_dense) -- pre-factored k-phase exp table
    E_r,                 # (n_uniq_n, n_t_dense) -- pre-factored n-phase exp table
    m_inv_b, k_inv_b, n_inv_b,   # (batch,) int32 -- indices into exp tables
):
    # Cubic Horner evaluation of mode amplitudes -- no linear weight w.
    # scipy c layout: c[0]=cubic, c[1]=quad, c[2]=linear, c[3]=constant
    teuk_dense = spline_coeffs_b[0, inds]
    teuk_dense = spline_coeffs_b[1, inds] + dx[:, None] * teuk_dense
    teuk_dense = spline_coeffs_b[2, inds] + dx[:, None] * teuk_dense
    teuk_dense = spline_coeffs_b[3, inds] + dx[:, None] * teuk_dense  # (n_t, batch)

    # Phase factor by gather + multiply -- no exp call here.
    E  = (E_phi[m_inv_b] * E_theta[k_inv_b] * E_r[n_inv_b]).T  # (n_t, batch)

    T  = teuk_dense * E
    w1 = T @ ylm_pos_b
    w2 = jnp.conj(T) @ coeff_neg_b
    return w1 + w2


@jax.jit
def _eval_bezier_phases_on_dense(t_dense, knot_t, coeff_3d):
    """DOPR853 degree-7 Bezier evaluation -- matches FEW's InterpolatedModeSum kernel.

    `coeff_3d` is FEW's `few_traj.integrator_spline_phase_coeff` with shape
    `(N_intervals, 3, 8)`. Axis-1 ordering is [Phi_phi, Phi_theta, Phi_r],
    axis-2 holds the 8 Bezier coefficients per interval, identical to the
    DOPR853 native dense-output polynomial used inside FEW.

    Formula (from interpolate.cu line 642):
        s  = (t - knot_t[i]) / (knot_t[i+1] - knot_t[i])
        s1 = 1 - s
        P(s) = c0 + s*(c1 + s1*(c2 + s*(c3 + s1*(c4 + s*(c5 + s1*(c6 + s*c7))))))
    """
    inds = jnp.clip(
        jnp.searchsorted(knot_t, t_dense, side="right") - 1,
        0, knot_t.size - 2,
    )
    s  = (t_dense - knot_t[inds]) / (knot_t[inds + 1] - knot_t[inds])
    s1 = 1.0 - s
    # Gather all 8 coeffs for all 3 phases at every dense time -- (n_t, 3, 8)
    c = coeff_3d[inds]
    # Horner-Bezier evaluation along the coefficient axis (inside-out)
    out = c[..., 7]
    out = c[..., 6] + s[:,  None] * out
    out = c[..., 5] + s1[:, None] * out
    out = c[..., 4] + s[:,  None] * out
    out = c[..., 3] + s1[:, None] * out
    out = c[..., 2] + s[:,  None] * out
    out = c[..., 1] + s1[:, None] * out
    out = c[..., 0] + s[:,  None] * out   # (n_t, 3)
    return out[:, 0], out[:, 1], out[:, 2]


def Interp_and_modesum_WFbuilder(
    time, teuk_modes, ylms, ls, ms, ks, ns,
    Phi_phi_tr, Phi_theta_tr, Phi_r_tr,
    t_dense,
    sum_batch_size=100,
    phase_spline_t=None,
    phase_spline_coeff=None,
):
    """Mode-batched fused interp + direct mode sum.

    Amplitudes: not-a-knot cubic spline (matches FEW's amp interpolation).
    Phases:
       - If `phase_spline_t` and `phase_spline_coeff` are provided, use FEW's
         DOPR853 degree-7 Bezier dense output. This matches FEW's
         `InterpolatedModeSum` to machine precision.  Pass:
              phase_spline_t     = few_traj.integrator_spline_t        # (N+1,)
              phase_spline_coeff = few_traj.integrator_spline_phase_coeff  # (N, 3, 8)
         right after calling few_traj(...) with the same (m1, m2, a, p0, e0, x0, T).
       - Otherwise, use a not-a-knot cubic spline on Phi_*_tr -- faster to
         construct but the phase between sparse knots drifts ~1e-3 rad
         relative to DOPR853, which is what causes the empirical "2i factor"
         observed when comparing against FEW's GenerateEMRIWaveform.

    All the speed optimisations from v3 are retained:
       - cubic spline for amplitudes (gather + Horner inside the JIT kernel)
       - exp factorisation: only ~20 complex exp() calls instead of n_m * n_t
       - mode-batched GEMV reductions
    """
    n_m  = int(ls.shape[0])
    t_np = np.array(time)

    # Fit cubic spline to amplitudes once (FEW also uses cubic for amps).
    spline_teuk = _fit_spline(t_np, np.array(teuk_modes))   # (4, n_sparse-1, n_m)

    # Bracket indices and actual time offset for amplitude eval.
    inds, w  = _bracket(time, t_dense)
    h_sparse = time[1:] - time[:-1]
    dx       = w * h_sparse[inds]

    # ------------------------------------------------------------------
    # Phase interpolation: either FEW DOPR853 Bezier (preferred) or cubic.
    # ------------------------------------------------------------------
    if phase_spline_t is not None and phase_spline_coeff is not None:
        knot_t_jax  = jnp.asarray(np.array(phase_spline_t),     dtype=jnp.float64)
        coeff_3d_jax = jnp.asarray(np.array(phase_spline_coeff), dtype=jnp.float64)
        Phi_phi_d, Phi_theta_d, Phi_r_d = _eval_bezier_phases_on_dense(
            t_dense, knot_t_jax, coeff_3d_jax,
        )
    else:
        # cubic spline fallback (slightly less accurate, no FEW state needed)
        phases_np = np.stack([np.array(Phi_phi_tr),
                              np.array(Phi_theta_tr),
                              np.array(Phi_r_tr)], axis=1)
        spline_phases = _fit_spline(t_np, phases_np)
        phases_dense = _eval_cubic(spline_phases, inds, dx)
        Phi_phi_d, Phi_theta_d, Phi_r_d = (
            phases_dense[:, 0], phases_dense[:, 1], phases_dense[:, 2]
        )

    # Ylm coefficient vectors (unchanged).
    ylm_pos   = ylms[:n_m]
    ylm_neg   = ylms[n_m:2 * n_m]
    coeff_neg = ((ms > 0) * ((-1.0) ** ls)).astype(ylm_neg.dtype) * ylm_neg

    # Exp factorisation (one exp per unique quantum number, not per mode).
    unique_ms, m_inv = np.unique(np.array(ms), return_inverse=True)
    unique_ks, k_inv = np.unique(np.array(ks), return_inverse=True)
    unique_ns, n_inv = np.unique(np.array(ns), return_inverse=True)

    E_phi   = jnp.exp(-1j * jnp.asarray(unique_ms)[:, None] * Phi_phi_d[None, :])
    E_theta = jnp.exp(-1j * jnp.asarray(unique_ks)[:, None] * Phi_theta_d[None, :])
    E_r     = jnp.exp(-1j * jnp.asarray(unique_ns)[:, None] * Phi_r_d[None, :])

    m_inv_j = jnp.asarray(m_inv, dtype=jnp.int32)
    k_inv_j = jnp.asarray(k_inv, dtype=jnp.int32)
    n_inv_j = jnp.asarray(n_inv, dtype=jnp.int32)

    out_wf = jnp.zeros(t_dense.shape[0], dtype=jnp.complex128)

    for i in range(0, n_m, sum_batch_size):
        i_end = min(i + sum_batch_size, n_m)
        out_wf = out_wf + _interp_modesum_batch_cubic(
            inds, dx, spline_teuk[:, :, i:i_end],
            ylm_pos[i:i_end], coeff_neg[i:i_end],
            E_phi, E_theta, E_r,
            m_inv_j[i:i_end], k_inv_j[i:i_end], n_inv_j[i:i_end],
        )

    return np.asarray(out_wf)


---
## Pipeline functions

Each returns `(h_plus, h_cross)` as numpy arrays on the dense grid (cadence `dt`).  Output length is whatever the trajectory + dense grid yields; the comparison loop truncates to the common length before computing the mismatch.


In [7]:
def jax_pipeline_cubic(m1, m2, a, p0, e0, x0, dist, theta_S, phi_S, T, dt):
    '''Fully-JAX pipeline: fewtrax trajectory + JAX amps + JAX mode selection
    + cubic-spline phase.  No FEW state required.
    '''
    theta_src, phi_src = _get_viewing_angles(theta_S, phi_S, 0.0, 0.0)

    # 1. Sparse trajectory (fewtrax JAX, dense_steps=80 keeps the JIT shape fixed).
    t, p, e, Pp, Pt, Pr = traj_jax(
        p0=p0, e0=e0, a=a, T=T, M=m1, mu=m2, dense_steps=80,
    )

    # 2. Amplitudes on the sparse trajectory (all 6993 modes).
    teuk_all = np.asarray(amp_jax_full.evaluate_trajectory(
        a, jnp.asarray(p), jnp.asarray(e),
    ))

    # 3. Per-mode Ylms (FEW's GetYlms, same generator FEW's selector uses).
    yp = model.ylm_gen(l_full,  m_full, theta_src, phi_src)
    yn = model.ylm_gen(l_full, -m_full, theta_src, phi_src)
    yp = yp.get() if hasattr(yp, 'get') else np.asarray(yp)
    yn = yn.get() if hasattr(yn, 'get') else np.asarray(yn)

    # 4. FEW-compatible JAX mode selection (one jitted GPU kernel).
    keep, _ = amp_jax_full.select_modes_few(
        a, p, e, yp, yn,
        threshold=mode_selection_threshold,
        teuk_modes=teuk_all,
    )

    teuk_sel = jnp.asarray(teuk_all[:, keep])
    ls = jnp.asarray(l_full[keep]); ms = jnp.asarray(m_full[keep])
    ns = jnp.asarray(n_full[keep]); ks = jnp.zeros_like(ls)
    ylms_sel = jnp.concatenate([jnp.asarray(yp[keep]), jnp.asarray(yn[keep])])

    # 5. Dense grid + waveform build with cubic-spline phases.
    n_pts   = int((float(t[-1]) - float(t[0])) / dt) + 1
    t_dense = float(t[0]) + dt * np.arange(n_pts)

    h = Interp_and_modesum_WFbuilder(
        t, teuk_sel, ylms_sel, ls, ms, ks, ns,
        Phi_phi_tr=Pp, Phi_theta_tr=Pt, Phi_r_tr=Pr,
        t_dense=t_dense,
        sum_batch_size=100,
    )

    scale = dist * Gpc / (m2 * MRSUN_SI)
    h = h / scale
    return -np.real(h), np.imag(h)


In [8]:
def jax_pipeline_dopr(m1, m2, a, p0, e0, x0, dist, theta_S, phi_S, T, dt):
    '''JAX waveform with FEW's DOPR853 Bezier phase interpolation.  Needs the
    FEW trajectory because the dense-output Bezier coefficients live on
    `few_traj.integrator_spline_*`.
    '''
    theta_src, phi_src = _get_viewing_angles(theta_S, phi_S, 0.0, 0.0)

    # 1. FEW trajectory (populates few_traj.integrator_spline_* state).
    t, p, e, x, Pp, Pt, Pr = few_traj(m1, m2, a, p0, e0, x0, T=T)

    # 2. Amplitudes on the sparse trajectory (JAX, all modes).
    teuk_all = np.asarray(amp_jax_full.evaluate_trajectory(
        a, jnp.asarray(p), jnp.asarray(e),
    ))

    # 3. Per-mode Ylms.
    yp = model.ylm_gen(l_full,  m_full, theta_src, phi_src)
    yn = model.ylm_gen(l_full, -m_full, theta_src, phi_src)
    yp = yp.get() if hasattr(yp, 'get') else np.asarray(yp)
    yn = yn.get() if hasattr(yn, 'get') else np.asarray(yn)

    # 4. FEW-compatible JAX mode selection.
    keep, _ = amp_jax_full.select_modes_few(
        a, p, e, yp, yn,
        threshold=mode_selection_threshold,
        teuk_modes=teuk_all,
    )

    teuk_sel = jnp.asarray(teuk_all[:, keep])
    ls = jnp.asarray(l_full[keep]); ms = jnp.asarray(m_full[keep])
    ns = jnp.asarray(n_full[keep]); ks = jnp.zeros_like(ls)
    ylms_sel = jnp.concatenate([jnp.asarray(yp[keep]), jnp.asarray(yn[keep])])

    # 5. Dense grid + waveform with DOPR853 Bezier phases.
    n_pts   = int((float(t[-1]) - float(t[0])) / dt) + 1
    t_dense = float(t[0]) + dt * np.arange(n_pts)

    h = Interp_and_modesum_WFbuilder(
        t, teuk_sel, ylms_sel, ls, ms, ks, ns,
        Phi_phi_tr=Pp, Phi_theta_tr=Pt, Phi_r_tr=Pr,
        t_dense=t_dense,
        sum_batch_size=100,
        phase_spline_t     = few_traj.integrator_spline_t,
        phase_spline_coeff = few_traj.integrator_spline_phase_coeff,
    )

    scale = dist * Gpc / (m2 * MRSUN_SI)
    h = h / scale
    return -np.real(h), np.imag(h)


In [9]:
def few_pipeline(m1, m2, a, p0, e0, x0, dist, theta_S, phi_S, T, dt):
    '''FEW reference via GenerateEMRIWaveform; same conventions as
    cell `b22903b5` of EMRI_wf_JAX_updated.ipynb.'''
    hp, hc = waveform_gen(
        m1, m2, a, float(p0), float(e0), x0, dist,
        theta_S, phi_S, 0.0, 0.0,
        0.0, 0.0, 0.0,
        T=T, dt=dt,
    )
    hp = hp.get() if hasattr(hp, 'get') else np.asarray(hp)
    hc = hc.get() if hasattr(hc, 'get') else np.asarray(hc)
    return hp, hc


---
## Physical priors and parameter sampling

Default priors (edit `sample_valid_params` to taste):

| param | distribution |
|---|---|
| `m1`   | log-uniform in `[1e5, 3.16e6] M_sun`  (log10 ∈ [5.0, 6.5]) |
| `m2`   | log-uniform in `[5, 31.6] M_sun`      (log10 ∈ [0.7, 1.5]) |
| `a`    | uniform in `[0.0, 0.95]` (prograde) |
| `e0`   | uniform in `[0.0, 0.4]` |
| `p0`   | chosen via `get_p_at_t` so the system plunges at `T_plunge = T_obs · (1 + δ)` with `δ ∼ U[0.05, 0.5]` |
| `x0`   | `1.0` (equatorial prograde — the only case the model supports) |
| `θ_S`  | uniform in `cos θ_S ∈ [−1, 1]` |
| `φ_S`  | uniform in `[0, 2π]` |
| `dist` | `1.0 Gpc` (irrelevant for the mismatch) |


In [10]:
def sample_valid_params(N, T_obs, seed=0, max_attempts_per_sample=10):
    '''Draw N parameter sets and pick p0 with get_p_at_t so the inspiral
    plunges between 1.05*T_obs and 1.5*T_obs — guarantees a full T_obs waveform.

    Returns a list of dicts; entries where get_p_at_t fails are skipped.
    '''
    rng = np.random.default_rng(seed)
    samples = []
    attempts = 0
    while len(samples) < N and attempts < N * max_attempts_per_sample:
        attempts += 1
        m1     = 10.0 ** rng.uniform(5.0, 6.5)
        m2     = 10.0 ** rng.uniform(0.7, 1.5)
        a      = rng.uniform(0.0, 0.95)
        e0     = rng.uniform(0.0, 0.4)
        x0     = 1.0
        T_plunge = T_obs * (1.0 + rng.uniform(0.05, 0.5))
        try:
            p0 = float(get_p_at_t(
                few_traj,
                T_plunge,
                traj_args=[m1, m2, a, e0, x0],
                bounds=[None, None],
            ))
        except Exception:
            continue
        theta_S = float(np.arccos(rng.uniform(-1.0, 1.0)))
        phi_S   = float(rng.uniform(0.0, 2.0 * np.pi))
        samples.append(dict(
            m1=float(m1), m2=float(m2), a=float(a),
            p0=p0, e0=float(e0), x0=x0,
            theta_S=theta_S, phi_S=phi_S,
            T_plunge=T_plunge,
        ))
        if len(samples) % 10 == 0:
            print(f"Drew {len(samples)} valid parameter sets in {attempts} attempts.")
    print(f"Drew {len(samples)} valid parameter sets in {attempts} attempts.")
    return samples


---
## Warm-up

One call of each pipeline to compile every JIT kernel we'll need.  Subsequent calls in the loop reuse the cached compilations when shapes match — JAX/XLA only recompiles when a new shape combination appears.


In [11]:
T_OBS = 2.0      # waveform length, years
DT    = 10.0     # cadence, seconds
DIST  = 1.0      # Gpc (irrelevant for mismatch)

_warm = sample_valid_params(1, T_OBS, seed=1234)[0]
print("Warm-up params:", {k: f"{v:.3g}" if isinstance(v, float) else v for k, v in _warm.items()})

print("Warming JAX cubic …")
tic = time.perf_counter()
_ = jax_pipeline_cubic(_warm['m1'], _warm['m2'], _warm['a'], _warm['p0'], _warm['e0'], _warm['x0'],
                       DIST, _warm['theta_S'], _warm['phi_S'], T_OBS, DT)
print(f"  cold call: {time.perf_counter()-tic:.2f} s")

print("Warming JAX DOPR …")
tic = time.perf_counter()
_ = jax_pipeline_dopr(_warm['m1'], _warm['m2'], _warm['a'], _warm['p0'], _warm['e0'], _warm['x0'],
                      DIST, _warm['theta_S'], _warm['phi_S'], T_OBS, DT)
print(f"  cold call: {time.perf_counter()-tic:.2f} s")

print("Warming FEW reference …")
tic = time.perf_counter()
_ = few_pipeline(_warm['m1'], _warm['m2'], _warm['a'], _warm['p0'], _warm['e0'], _warm['x0'],
                 DIST, _warm['theta_S'], _warm['phi_S'], T_OBS, DT)
print(f"  cold call: {time.perf_counter()-tic:.2f} s")


Drew 1 valid parameter sets in 1 attempts.
Warm-up params: {'m1': '2.92e+06', 'm2': '10.1', 'a': '0.877', 'p0': '5.33', 'e0': '0.105', 'x0': '1', 'theta_S': '2.44', 'phi_S': '1.52', 'T_plunge': '2.39'}
Warming JAX cubic …
  cold call: 22.99 s
Warming JAX DOPR …
  cold call: 4.74 s
Warming FEW reference …
  cold call: 1.54 s


---
## Per-piece timing (warm: reuses cached JIT compiles)

Same parameter set as the warm-up.  Every JAX kernel is already compiled from the previous cell, so these timings measure steady-state cost — what each piece will contribute inside the comparison loop whenever the shape signature is reused.


In [13]:
T  = T_OBS
dt = DT

_p = _warm
m1, m2 = _p['m1'], _p['m2']
a, p0, e0, x0 = _p['a'], _p['p0'], _p['e0'], _p['x0']
theta_S, phi_S = _p['theta_S'], _p['phi_S']
dist = DIST

def _wait(x):
    """Block until a JAX array is materialised on its device."""
    if hasattr(x, "block_until_ready"):
        x.block_until_ready()
    return x

def _profile_jax(label, phase_mode, theta_src, phi_src):
    """Run one JAX pipeline with finer-grained per-piece timing.

    Splits step 2 (amplitudes) into GPU-compute vs GPU->host sync, and step
    4 (mode selection) into host->GPU push + jitted scoring + host
    post-processing.  This isolates true compute cost from device-transfer
    overhead.
    """
    print("=" * 72)
    print(f"{label}  (warm)")
    print("=" * 72)

    # ---- 1. Sparse trajectory ------------------------------------------------
    tic = time.perf_counter()
    if phase_mode == "cubic":
        t, p, e, Pp, Pt, Pr = traj_jax(
            p0=p0, e0=e0, a=a, T=T, M=m1, mu=m2, dense_steps=80,
        )
        _wait(t)
    else:
        t, p, e, _x, Pp, Pt, Pr = few_traj(m1, m2, a, p0, e0, x0, T=T)
    t_traj = time.perf_counter() - tic
    print(f"  1. sparse trajectory                                : {t_traj*1e3:9.2f} ms   n_sparse={len(t)}")

    # ---- 2a. JAX amplitudes — GPU compute only -------------------------------
    p_j = jnp.asarray(p); e_j = jnp.asarray(e)
    tic = time.perf_counter()
    teuk_all_j = amp_jax_full.evaluate_trajectory(a, p_j, e_j)
    _wait(teuk_all_j)
    t_amps_gpu = time.perf_counter() - tic
    print(f"  2a. amplitudes (GPU compute only, block_until_ready): {t_amps_gpu*1e3:9.2f} ms   shape={teuk_all_j.shape}")

    # ---- 2b. GPU -> host transfer of the (N_t, n_modes) table ---------------
    tic = time.perf_counter()
    teuk_all_np = np.asarray(teuk_all_j)
    t_amps_dl = time.perf_counter() - tic
    nbytes = teuk_all_np.nbytes / 1e6
    print(f"  2b. amplitudes (GPU -> host {nbytes:.1f} MB)                    : {t_amps_dl*1e3:9.2f} ms")

    # ---- 3. Ylms ------------------------------------------------------------
    tic = time.perf_counter()
    yp = model.ylm_gen(l_full,  m_full, theta_src, phi_src)
    yn = model.ylm_gen(l_full, -m_full, theta_src, phi_src)
    yp = yp.get() if hasattr(yp, "get") else np.asarray(yp)
    yn = yn.get() if hasattr(yn, "get") else np.asarray(yn)
    t_ylm = time.perf_counter() - tic
    print(f"  3. Ylms (per-mode, FEW GetYlms)                     : {t_ylm*1e3:9.2f} ms")

    # ---- 4. Mode selection (split into pieces) ------------------------------
    # 4a. Push numpy teuk back to GPU + score (one JIT kernel)
    tic = time.perf_counter()
    keep, _ = amp_jax_full.select_modes_few(
        a, p, e, yp, yn,
        threshold=mode_selection_threshold,
        teuk_modes=teuk_all_np,
    )
    t_sel = time.perf_counter() - tic
    print(f"  4. mode selection (incl. host<->GPU)                : {t_sel*1e3:9.2f} ms   n_kept={len(keep)}")

    # ---- 5. Gather + dense grid ---------------------------------------------
    tic = time.perf_counter()
    teuk_sel = jnp.asarray(teuk_all_np[:, keep])
    ls = jnp.asarray(l_full[keep]); ms = jnp.asarray(m_full[keep])
    ns = jnp.asarray(n_full[keep]); ks = jnp.zeros_like(ls)
    ylms_sel = jnp.concatenate([jnp.asarray(yp[keep]), jnp.asarray(yn[keep])])
    n_pts = int((float(t[-1]) - float(t[0])) / dt) + 1
    t_dense = float(t[0]) + dt * np.arange(n_pts)
    t_prep = time.perf_counter() - tic
    print(f"  5. gather kept modes + build dense grid             : {t_prep*1e3:9.2f} ms   n_t_dense={n_pts}")

    # ---- 6. Interp + mode-sum (the heavy waveform kernel) -------------------
    tic = time.perf_counter()
    if phase_mode == "cubic":
        h = Interp_and_modesum_WFbuilder(
            t, teuk_sel, ylms_sel, ls, ms, ks, ns,
            Phi_phi_tr=Pp, Phi_theta_tr=Pt, Phi_r_tr=Pr,
            t_dense=t_dense, sum_batch_size=100,
        )
    else:
        h = Interp_and_modesum_WFbuilder(
            t, teuk_sel, ylms_sel, ls, ms, ks, ns,
            Phi_phi_tr=Pp, Phi_theta_tr=Pt, Phi_r_tr=Pr,
            t_dense=t_dense, sum_batch_size=100,
            phase_spline_t     = few_traj.integrator_spline_t,
            phase_spline_coeff = few_traj.integrator_spline_phase_coeff,
        )
    t_wfb = time.perf_counter() - tic
    print(f"  6. Interp_and_modesum_WFbuilder                     : {t_wfb*1e3:9.2f} ms")

    # ---- 7. Distance scaling + (h+, hx) split --------------------------------
    tic = time.perf_counter()
    scale = dist * Gpc / (m2 * MRSUN_SI)
    h = h / scale
    hp, hc = -np.real(h), np.imag(h)
    t_scale = time.perf_counter() - tic
    print(f"  7. scale + split (h+, hx)                           : {t_scale*1e3:9.2f} ms")

    total = t_traj + t_amps_gpu + t_amps_dl + t_ylm + t_sel + t_prep + t_wfb + t_scale
    print(f"  TOTAL                                               : {total*1e3:9.2f} ms")
    print()
    return hp, hc, total

theta_src, phi_src = _get_viewing_angles(theta_S, phi_S, 0.0, 0.0)
hp_cub, hc_cub, t_cubic_total = _profile_jax(
    "JAX cubic (fewtrax traj + cubic phase, fully JAX)", "cubic", theta_src, phi_src,
)
hp_dop, hc_dop, t_dopr_total  = _profile_jax(
    "JAX DOPR  (FEW traj + DOPR853 Bezier phase)",       "dopr",  theta_src, phi_src,
)

# FEW reference for context
print("=" * 72)
print("FEW reference  (single call)")
print("=" * 72)
tic = time.perf_counter()
hp_few, hc_few = few_pipeline(m1, m2, a, p0, e0, x0, dist, theta_S, phi_S, T, dt)
t_few_total = time.perf_counter() - tic
print(f"  GenerateEMRIWaveform                                : {t_few_total*1e3:9.2f} ms   n_dense={hp_few.size}")
print()
print("Warm-state speed-up:")
print(f"  FEW / JAX cubic = {t_few_total / t_cubic_total:7.2f}x")
print(f"  FEW / JAX DOPR  = {t_few_total / t_dopr_total :7.2f}x")
print()
n_common = min(hp_few.size, hp_cub.size, hp_dop.size)
print("Mismatch on this warm-up waveform:")
print(f"  FEW vs JAX cubic  h+ : {get_mismatch(hp_few[:n_common], hp_cub[:n_common]):.3e}")
print(f"  FEW vs JAX cubic  hx : {get_mismatch(hc_few[:n_common], hc_cub[:n_common]):.3e}")
print(f"  FEW vs JAX DOPR   h+ : {get_mismatch(hp_few[:n_common], hp_dop[:n_common]):.3e}")
print(f"  FEW vs JAX DOPR   hx : {get_mismatch(hc_few[:n_common], hc_dop[:n_common]):.3e}")


JAX cubic (fewtrax traj + cubic phase, fully JAX)  (warm)
  1. sparse trajectory                                :    129.26 ms   n_sparse=80
  2a. amplitudes (GPU compute only, block_until_ready):    488.36 ms   shape=(80, 6993)
  2b. amplitudes (GPU -> host 9.0 MB)                    :      1.67 ms
  3. Ylms (per-mode, FEW GetYlms)                     :      3.05 ms
  4. mode selection (incl. host<->GPU)                :      4.85 ms   n_kept=71
  5. gather kept modes + build dense grid             :     23.30 ms   n_t_dense=6311630
  6. Interp_and_modesum_WFbuilder                     :     92.43 ms
  7. scale + split (h+, hx)                           :     41.54 ms
  TOTAL                                               :    784.46 ms

JAX DOPR  (FEW traj + DOPR853 Bezier phase)  (warm)
  1. sparse trajectory                                :     16.58 ms   n_sparse=15
  2a. amplitudes (GPU compute only, block_until_ready):    484.65 ms   shape=(15, 6993)
  2b. amplitudes (GPU -> host

---
## Speed + mismatch loop on 500 waveforms

For each parameter set we run all three pipelines, time them, and compute the mismatch (FEW vs JAX-cubic, FEW vs JAX-DOPR) on `h+` and `h×`.  Waveforms are not stored — only timings and mismatches.  Failures are logged and skipped.


In [ ]:
def run_one(p, T, dt):
    dist = DIST
    out = {}

    tic = time.perf_counter()
    hp_few, hc_few = few_pipeline(
        p['m1'], p['m2'], p['a'], p['p0'], p['e0'], p['x0'],
        dist, p['theta_S'], p['phi_S'], T, dt,
    )
    out['t_few'] = time.perf_counter() - tic
    print(f"FEW pipeline done in {out['t_few']:.2f}s")

    tic = time.perf_counter()
    hp_cub, hc_cub = jax_pipeline_cubic(
        p['m1'], p['m2'], p['a'], p['p0'], p['e0'], p['x0'],
        dist, p['theta_S'], p['phi_S'], T, dt,
    )
    out['t_jax_cubic'] = time.perf_counter() - tic
    print(f"JAX-cubic pipeline done in {out['t_jax_cubic']:.2f}s")

    tic = time.perf_counter()
    hp_dop, hc_dop = jax_pipeline_dopr(
        p['m1'], p['m2'], p['a'], p['p0'], p['e0'], p['x0'],
        dist, p['theta_S'], p['phi_S'], T, dt,
    )
    out['t_jax_dopr'] = time.perf_counter() - tic

    print('waveform generation done, computing mismatches…')
    n = min(hp_few.size, hp_cub.size, hp_dop.size)
    out['mm_cubic_hp'] = float(get_mismatch(hp_few[:n], hp_cub[:n]))
    out['mm_cubic_hc'] = float(get_mismatch(hc_few[:n], hc_cub[:n]))
    out['mm_dopr_hp']  = float(get_mismatch(hp_few[:n], hp_dop[:n]))
    out['mm_dopr_hc']  = float(get_mismatch(hc_few[:n], hc_dop[:n]))
    out['n_dense']     = n
    return out

N_TOTAL = 100
#params_list = sample_valid_params(N_TOTAL, T_OBS, seed=42)
print(f"Running pipelines on {len(params_list)} parameter sets …")

records = []
fails   = 0
t_start = time.perf_counter()

for i, p in enumerate(params_list):
    try:
        r = run_one(p, T_OBS, DT)
        r.update(p)
        records.append(r)
    except Exception as exc:
        fails += 1
        print(f"  [{i+1}/{len(params_list)}] FAILED: {exc}")
        continue
    if (i + 1) % 10 == 0 or (i + 1) == len(params_list):
        elapsed = time.perf_counter() - t_start
        eta = elapsed / (i + 1) * (len(params_list) - (i + 1))
        print(f"  [{i+1}/{len(params_list)}]  "
              f"FEW={r['t_few']:6.2f}s  JAX-cub={r['t_jax_cubic']:5.2f}s  "
              f"JAX-dop={r['t_jax_dopr']:5.2f}s  mm-cub={r['mm_cubic_hp']:.2e}  "
              f"mm-dop={r['mm_dopr_hp']:.2e}   elapsed={elapsed/60:.1f}m  eta={eta/60:.1f}m")

print(f"\nDone — {len(records)} successful, {fails} failed, total {(time.perf_counter()-t_start)/60:.1f} min.")
np.savez('emri_compare_results.npz', records=records)


Drew 10 valid parameter sets in 10 attempts.
Drew 20 valid parameter sets in 20 attempts.
Drew 30 valid parameter sets in 30 attempts.
Drew 40 valid parameter sets in 40 attempts.
Drew 50 valid parameter sets in 50 attempts.
Drew 60 valid parameter sets in 60 attempts.
Drew 70 valid parameter sets in 70 attempts.
Drew 80 valid parameter sets in 80 attempts.
Drew 90 valid parameter sets in 90 attempts.
Drew 100 valid parameter sets in 100 attempts.
Drew 110 valid parameter sets in 110 attempts.
Drew 120 valid parameter sets in 120 attempts.
Drew 130 valid parameter sets in 130 attempts.
Drew 140 valid parameter sets in 140 attempts.
Drew 150 valid parameter sets in 150 attempts.
Drew 160 valid parameter sets in 160 attempts.
Drew 170 valid parameter sets in 170 attempts.
Drew 180 valid parameter sets in 180 attempts.
Drew 190 valid parameter sets in 190 attempts.
Drew 200 valid parameter sets in 200 attempts.
Drew 210 valid parameter sets in 210 attempts.
Drew 220 valid parameter sets i

E0528 08:31:07.239579 3555358 cuda_executor.cc:1273] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 12.62GiB (13549469696 bytes) of device memory: : CUDA_ERROR_OUT_OF_MEMORY: out of memory


JAX-cubic pipeline done in 1.68s
waveform generation done, computing mismatches…
FEW pipeline done in 1.12s
JAX-cubic pipeline done in 1.48s
waveform generation done, computing mismatches…
FEW pipeline done in 0.08s
JAX-cubic pipeline done in 2.21s
waveform generation done, computing mismatches…
FEW pipeline done in 0.08s
JAX-cubic pipeline done in 1.88s
waveform generation done, computing mismatches…
FEW pipeline done in 0.06s
JAX-cubic pipeline done in 1.22s
waveform generation done, computing mismatches…
FEW pipeline done in 0.09s


KeyboardInterrupt: 

---
## Analysis


In [ ]:
import numpy as np
arr = lambda key: np.asarray([r[key] for r in records])

t_few = arr('t_few');     t_cub = arr('t_jax_cubic');     t_dop = arr('t_jax_dopr')
mm_cub_hp = arr('mm_cubic_hp');  mm_cub_hc = arr('mm_cubic_hc')
mm_dop_hp = arr('mm_dopr_hp');   mm_dop_hc = arr('mm_dopr_hc')

def _pct(x, p): return float(np.percentile(x, p))

print("Generation time (seconds)")
print(f"  FEW       : median={_pct(t_few, 50):6.2f}s  p10={_pct(t_few,10):6.2f}  p90={_pct(t_few,90):6.2f}")
print(f"  JAX cubic : median={_pct(t_cub, 50):6.2f}s  p10={_pct(t_cub,10):6.2f}  p90={_pct(t_cub,90):6.2f}")
print(f"  JAX DOPR  : median={_pct(t_dop, 50):6.2f}s  p10={_pct(t_dop,10):6.2f}  p90={_pct(t_dop,90):6.2f}")
print()
print("Speed-up vs FEW (median)")
print(f"  JAX cubic : {np.median(t_few/t_cub):6.1f}x")
print(f"  JAX DOPR  : {np.median(t_few/t_dop):6.1f}x")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
bins = np.linspace(0, max(t_few.max(), t_cub.max(), t_dop.max()) * 1.05, 40)
ax[0].hist(t_few, bins=bins, alpha=0.55, label='FEW',       color='C0')
ax[0].hist(t_cub, bins=bins, alpha=0.55, label='JAX cubic', color='C1')
ax[0].hist(t_dop, bins=bins, alpha=0.55, label='JAX DOPR',  color='C2')
ax[0].set_xlabel('generation time [s]'); ax[0].set_ylabel('count'); ax[0].legend(); ax[0].set_title('Wall time per waveform')

# Speed-up scatter vs M = m1+m2 to see if scaling is uniform
M = arr('m1') + arr('m2')
ax[1].loglog(M, t_few/t_cub, '.', alpha=0.5, label='JAX cubic')
ax[1].loglog(M, t_few/t_dop, '.', alpha=0.5, label='JAX DOPR')
ax[1].set_xlabel('m1 + m2  [M_sun]'); ax[1].set_ylabel('FEW / JAX (speed-up)'); ax[1].legend()
ax[1].set_title('Speed-up vs total mass')
plt.tight_layout(); plt.show()


In [ ]:
def _stats(label, x):
    print(f"  {label:18s}  median={np.median(x):.3e}  p90={np.percentile(x,90):.3e}  max={x.max():.3e}")

print("Mismatch vs FEW reference (h_plus)")
_stats('JAX cubic h+', mm_cub_hp)
_stats('JAX DOPR  h+', mm_dop_hp)
print("Mismatch vs FEW reference (h_cross)")
_stats('JAX cubic hx', mm_cub_hc)
_stats('JAX DOPR  hx', mm_dop_hc)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
bins = np.logspace(-13, -2, 50)
ax[0].hist(mm_cub_hp, bins=bins, alpha=0.55, label='JAX cubic h+', color='C1')
ax[0].hist(mm_dop_hp, bins=bins, alpha=0.55, label='JAX DOPR  h+', color='C2')
ax[0].set_xscale('log'); ax[0].set_xlabel('mismatch'); ax[0].set_ylabel('count'); ax[0].legend()
ax[0].set_title('Mismatch vs FEW — h+')
ax[1].hist(mm_cub_hc, bins=bins, alpha=0.55, label='JAX cubic hx', color='C1')
ax[1].hist(mm_dop_hc, bins=bins, alpha=0.55, label='JAX DOPR  hx', color='C2')
ax[1].set_xscale('log'); ax[1].set_xlabel('mismatch'); ax[1].legend()
ax[1].set_title('Mismatch vs FEW — hx')
plt.tight_layout(); plt.show()

# 2-D: does mismatch depend on the source parameters?
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].loglog(M, mm_cub_hp, '.', alpha=0.5, label='cubic')
ax[0].loglog(M, mm_dop_hp, '.', alpha=0.5, label='DOPR')
ax[0].set_xlabel('m1 + m2  [M_sun]'); ax[0].set_ylabel('mismatch h+'); ax[0].legend()
ax[1].semilogy(arr('e0'), mm_cub_hp, '.', alpha=0.5, label='cubic')
ax[1].semilogy(arr('e0'), mm_dop_hp, '.', alpha=0.5, label='DOPR')
ax[1].set_xlabel('e0'); ax[1].set_ylabel('mismatch h+'); ax[1].legend()
plt.tight_layout(); plt.show()
